# Wizualizacja rynku koncertów muzycznych w Polsce

Notebook zawiera kompletne rozwiązanie zadania w **pandas** i **plotly**.  
Wykresy są interaktywne i mogą zostać później przeniesione do aplikacji Streamlit.


## 0. Generowanie danych

Poniższą komórkę uruchamiamy **raz**. Zapisuje ona plik `koncerty_polska.csv` w katalogu roboczym notebooka.


In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

np.random.seed(42)

n = 1200

miasta = {
    "Warszawa":   (52.2297, 21.0122, 1.00),
    "Kraków":     (50.0647, 19.9450, 0.75),
    "Wrocław":    (51.1079, 17.0385, 0.65),
    "Poznań":     (52.4064, 16.9252, 0.55),
    "Gdańsk":     (54.3520, 18.6466, 0.55),
    "Łódź":       (51.7592, 19.4560, 0.50),
    "Katowice":   (50.2649, 19.0238, 0.45),
    "Lublin":     (51.2465, 22.5684, 0.30),
    "Białystok":  (53.1325, 23.1688, 0.25),
    "Szczecin":   (53.4285, 14.5528, 0.35),
}

gatunki = ["rock", "pop", "hip-hop", "electronic", "jazz",
           "classical", "folk", "metal", "indie", "reggae"]

typy_obiektow = ["klub", "arena", "stadion", "festiwal", "teatr", "amfiteatr"]

kapacjety = {
    "klub": (200, 1500), "arena": (3000, 15000), "stadion": (20000, 70000),
    "festiwal": (10000, 80000), "teatr": (400, 2000), "amfiteatr": (1500, 8000),
}

cena_bazowa = {
    "rock": 150, "pop": 200, "hip-hop": 180, "electronic": 160, "jazz": 130,
    "classical": 110, "folk": 90, "metal": 140, "indie": 100, "reggae": 110,
}

cena_mnoznik = {
    "klub": 0.7, "arena": 1.3, "stadion": 1.8,
    "festiwal": 1.5, "teatr": 1.2, "amfiteatr": 1.0,
}

start_date = datetime(2024, 1, 1)
daty = [start_date + timedelta(days=int(d)) for d in np.random.randint(0, 730, n)]

wagi = np.array([miasta[m][2] for m in miasta])
miasto = np.random.choice(list(miasta.keys()), n, p=wagi / wagi.sum())
gatunek = np.random.choice(gatunki, n)
typ_obiektu = np.random.choice(
    typy_obiektow, n, p=[0.40, 0.15, 0.05, 0.10, 0.15, 0.15]
)

pojemnosc = np.array([np.random.randint(*kapacjety[t]) for t in typ_obiektu])
wypelnienie = np.clip(np.random.beta(5, 2, n), 0.15, 1.0)
sprzedane = (pojemnosc * wypelnienie).astype(int)

cena = np.array([
    cena_bazowa[g] * cena_mnoznik[t]
    for g, t in zip(gatunek, typ_obiektu)
])
cena = np.round(cena * np.random.uniform(0.7, 1.4, n), -1)
przychod = (cena * sprzedane).astype(int)

df = pd.DataFrame({
    "event_id": range(50001, 50001 + n),
    "data": daty,
    "miasto": miasto,
    "latitude": [miasta[m][0] for m in miasto],
    "longitude": [miasta[m][1] for m in miasto],
    "gatunek": gatunek,
    "typ_obiektu": typ_obiektu,
    "pojemnosc": pojemnosc,
    "bilety_sprzedane": sprzedane,
    "cena_biletu_pln": cena,
    "przychod_pln": przychod,
})

df.to_csv("koncerty_polska.csv", index=False)
print(f"Wygenerowano plik 'koncerty_polska.csv' — {len(df)} koncertów")

Wygenerowano plik 'koncerty_polska.csv' — 1200 koncertów


## 1. Wczytanie i wstępna eksploracja

In [2]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# `parse_dates` zapewnia właściwy typ kolumny daty.
df = pd.read_csv("koncerty_polska.csv", parse_dates=["data"])

print("Shape:", df.shape)
display(df.head())
print("\nTypy danych:")
display(df.dtypes.to_frame("dtype"))
print(f"\nLiczba unikalnych miast: {df['miasto'].nunique()}")
print(f"Liczba unikalnych gatunków: {df['gatunek'].nunique()}")

Shape: (1200, 11)


,event_id,data,miasto,latitude,longitude,gatunek,typ_obiektu,pojemnosc,bilety_sprzedane,cena_biletu_pln,przychod_pln
0,50001,2024-04-12,Poznań,52.4064,16.9252,hip-hop,klub,410,316,170.0,53720
1,50002,2025-03-11,Gdańsk,54.3520,18.6466,folk,arena,5799,2512,130.0,326560
2,50003,2024-09-27,Warszawa,52.2297,21.0122,classical,klub,634,414,100.0,41400
3,50004,2024-04-16,Katowice,50.2649,19.0238,metal,klub,979,735,100.0,73500
4,50005,2024-03-12,Poznań,52.4064,16.9252,metal,klub,1430,808,80.0,64640



Typy danych:


,dtype
event_id,int64
data,datetime64[ns]
miasto,object
latitude,float64
longitude,float64
gatunek,object
typ_obiektu,object
pojemnosc,int64
bilety_sprzedane,int64
cena_biletu_pln,float64



Liczba unikalnych miast: 10
Liczba unikalnych gatunków: 10


## 2. Łączny przychód według miasta

In [3]:
przychod_miasto = (
    df.groupby("miasto", as_index=False)["przychod_pln"]
      .sum()
      .sort_values("przychod_pln", ascending=False)
)

fig_przychod_miasto = px.bar(
    przychod_miasto,
    x="miasto",
    y="przychod_pln",
    text_auto=".3s",
    labels={
        "miasto": "Miasto",
        "przychod_pln": "Łączny przychód [PLN]",
    },
    title="Łączny przychód z koncertów według miasta",
)

fig_przychod_miasto.update_layout(
    xaxis_tickangle=-35,
    yaxis_tickformat=",.0f",
    hovermode="x unified",
)
fig_przychod_miasto.show()

## 3. Szereg czasowy — liczba koncertów

In [4]:
# Czytelna etykieta miesiąca w formacie RRRR-MM.
df["miesiac"] = df["data"].dt.to_period("M").astype(str)

koncerty_miesiecznie = (
    df.groupby("miesiac", as_index=False)
      .size()
      .rename(columns={"size": "liczba_koncertow"})
)

fig_miesiace = px.line(
    koncerty_miesiecznie,
    x="miesiac",
    y="liczba_koncertow",
    markers=True,
    labels={
        "miesiac": "Miesiąc",
        "liczba_koncertow": "Liczba koncertów",
    },
    title="Łączna liczba koncertów w poszczególnych miesiącach",
)
fig_miesiace.update_layout(xaxis_tickangle=-45)
fig_miesiace.show()

In [5]:
koncerty_miesiac_typ = (
    df.groupby(["miesiac", "typ_obiektu"], as_index=False)
      .size()
      .rename(columns={"size": "liczba_koncertow"})
)

fig_miesiace_typ = px.line(
    koncerty_miesiac_typ,
    x="miesiac",
    y="liczba_koncertow",
    color="typ_obiektu",
    markers=True,
    labels={
        "miesiac": "Miesiąc",
        "liczba_koncertow": "Liczba koncertów",
        "typ_obiektu": "Typ obiektu",
    },
    title="Miesięczna liczba koncertów według typu obiektu",
)
fig_miesiace_typ.update_layout(xaxis_tickangle=-45, legend_title_text="Typ obiektu")
fig_miesiace_typ.show()

## 4. Rozkład cen biletów i przychodów

In [6]:
# Porównanie różnych liczb koszyków histogramu.
for nbins in [20, 50, 100]:
    fig_test = px.histogram(
        df,
        x="cena_biletu_pln",
        nbins=nbins,
        labels={"cena_biletu_pln": "Cena biletu [PLN]"},
        title=f"Rozkład cen biletów — {nbins} koszyków",
    )
    fig_test.show()

# 50 koszyków zachowuje dobry kompromis między szczegółowością a czytelnością.
fig_hist_ceny = px.histogram(
    df,
    x="cena_biletu_pln",
    nbins=50,
    labels={"cena_biletu_pln": "Cena biletu [PLN]", "count": "Liczba koncertów"},
    title="Rozkład cen biletów (50 koszyków)",
)
fig_hist_ceny.show()

In [7]:
fig_box_przychod = px.box(
    df,
    x="typ_obiektu",
    y="przychod_pln",
    points="outliers",
    labels={
        "typ_obiektu": "Typ obiektu",
        "przychod_pln": "Przychód z koncertu [PLN]",
    },
    title="Przychód z koncertu według typu obiektu",
)
fig_box_przychod.update_layout(yaxis_tickformat=",.0f")
fig_box_przychod.show()

**Komentarz:** W tej próbce najwyższe przychody z pojedynczego koncertu generują **stadiony**.  
Średni przychód wynosi tam około **7 592 995 PLN**  a wysokie wartości wynikają przede wszystkim z dużej pojemności obiektów.


## 5. Zależność ceny, wypełnienia i pojemności

In [8]:
df["wypelnienie"] = df["bilety_sprzedane"] / df["pojemnosc"]

fig_scatter = px.scatter(
    df,
    x="cena_biletu_pln",
    y="wypelnienie",
    color="gatunek",
    size="pojemnosc",
    size_max=45,
    hover_data={
        "miasto": True,
        "typ_obiektu": True,
        "pojemnosc": ":,",
        "bilety_sprzedane": ":,",
        "cena_biletu_pln": ":.0f",
        "wypelnienie": ":.1%",
    },
    labels={
        "cena_biletu_pln": "Cena biletu [PLN]",
        "wypelnienie": "Wypełnienie sali",
        "gatunek": "Gatunek",
        "pojemnosc": "Pojemność",
    },
    title="Cena biletu a wypełnienie sali",
)
fig_scatter.update_yaxes(tickformat=".0%")
fig_scatter.show()

korelacja = df["cena_biletu_pln"].corr(df["wypelnienie"])
print(f"Korelacja Pearsona cena–wypełnienie: {korelacja:.3f}")

Korelacja Pearsona cena–wypełnienie: 0.009


**Komentarz:** Nie widać wyraźnej zależności liniowej między ceną biletu a wypełnieniem sali.  
Współczynnik korelacji Pearsona wynosi **0.009**, czyli jest bliski zeru. W tym syntetycznym zbiorze wypełnienie zostało wylosowane niezależnie od ceny.


## 6. Mapa polskich miast koncertowych

In [9]:
miasta_agregacja = (
    df.groupby(["miasto", "latitude", "longitude"], as_index=False)
      .agg(
          srednia_cena_biletu_pln=("cena_biletu_pln", "mean"),
          liczba_koncertow=("event_id", "count"),
          laczny_przychod_pln=("przychod_pln", "sum"),
      )
)

fig_mapa = px.scatter_mapbox(
    miasta_agregacja,
    lat="latitude",
    lon="longitude",
    size="liczba_koncertow",
    color="srednia_cena_biletu_pln",
    hover_name="miasto",
    hover_data={
        "liczba_koncertow": ":,",
        "srednia_cena_biletu_pln": ":.2f",
        "laczny_przychod_pln": ":,",
        "latitude": False,
        "longitude": False,
    },
    labels={
        "liczba_koncertow": "Liczba koncertów",
        "srednia_cena_biletu_pln": "Średnia cena biletu [PLN]",
        "laczny_przychod_pln": "Łączny przychód [PLN]",
    },
    mapbox_style="open-street-map",
    zoom=4.7,
    center={"lat": 52.0, "lon": 19.2},
    size_max=48,
    title="Koncerty w polskich miastach: skala, cena i przychód",
)
fig_mapa.update_layout(margin={"r": 0, "t": 55, "l": 0, "b": 0})
fig_mapa.show()

display(miasta_agregacja.sort_values("liczba_koncertow", ascending=False))

/tmp/ipykernel_2153/3689551678.py:10: DeprecationWarning:

*scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/



,miasto,latitude,longitude,srednia_cena_biletu_pln,liczba_koncertow,laczny_przychod_pln
7,Warszawa,52.2297,21.0122,154.688797,241,391973430
3,Kraków,50.0647,19.9450,152.611465,157,175778640
8,Wrocław,51.1079,17.0385,135.923077,130,125091800
1,Gdańsk,54.3520,18.6466,151.653543,127,192767710
9,Łódź,51.7592,19.4560,148.699187,123,181357180
5,Poznań,52.4064,16.9252,147.377049,122,170662360
2,Katowice,50.2649,19.0238,157.884615,104,137064460
6,Szczecin,53.4285,14.5528,156.666667,84,155743970
4,Lublin,51.2465,22.5684,156.250000,72,136115110
0,Białystok,53.1325,23.1688,142.500000,40,50603470


## 7. Kompozycja — podsumowanie w układzie 2 × 2

In [10]:
liczba_gatunek = (
    df.groupby("gatunek", as_index=False)
      .size()
      .rename(columns={"size": "liczba_koncertow"})
      .sort_values("liczba_koncertow", ascending=False)
)

fig_podsumowanie = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=(
        "Łączny przychód według miasta",
        "Liczba koncertów według gatunku",
        "Rozkład cen biletów",
        "Wypełnienie sali według typu obiektu",
    ),
)

fig_podsumowanie.add_trace(
    go.Bar(x=przychod_miasto["miasto"], y=przychod_miasto["przychod_pln"], name="Przychód"),
    row=1, col=1,
)
fig_podsumowanie.add_trace(
    go.Bar(x=liczba_gatunek["gatunek"], y=liczba_gatunek["liczba_koncertow"], name="Koncerty"),
    row=1, col=2,
)
fig_podsumowanie.add_trace(
    go.Histogram(x=df["cena_biletu_pln"], nbinsx=50, name="Ceny biletów"),
    row=2, col=1,
)
fig_podsumowanie.add_trace(
    go.Box(
        x=df["typ_obiektu"],
        y=df["wypelnienie"],
        name="Wypełnienie",
        boxmean=True,
    ),
    row=2, col=2,
)

fig_podsumowanie.update_xaxes(title_text="Miasto", tickangle=-35, row=1, col=1)
fig_podsumowanie.update_yaxes(title_text="Przychód [PLN]", tickformat=",.0f", row=1, col=1)
fig_podsumowanie.update_xaxes(title_text="Gatunek", tickangle=-35, row=1, col=2)
fig_podsumowanie.update_yaxes(title_text="Liczba koncertów", row=1, col=2)
fig_podsumowanie.update_xaxes(title_text="Cena biletu [PLN]", row=2, col=1)
fig_podsumowanie.update_yaxes(title_text="Liczba koncertów", row=2, col=1)
fig_podsumowanie.update_xaxes(title_text="Typ obiektu", tickangle=-25, row=2, col=2)
fig_podsumowanie.update_yaxes(title_text="Wypełnienie", tickformat=".0%", row=2, col=2)

fig_podsumowanie.update_layout(
    title_text="Podsumowanie rynku koncertów muzycznych w Polsce",
    height=850,
    showlegend=False,
)
fig_podsumowanie.show()

## 8. Wnioski

1. **Warszawa** ma najwięcej imprez — **241 koncertów** — i jednocześnie najwyższy łączny przychód w analizowanym zbiorze.
2. Najwyższą średnią cenę biletów mają **stadiony** (około **251 PLN**)  ponieważ dla tego typu obiektu zastosowano najwyższy mnożnik cenowy.
3. **Stadion** generują najwyższe przychody z pojedynczego wydarzenia  co wiąże się z ich dużą pojemnością.
4. Nie ma silnej sezonowości: liczba koncertów oscyluje wokół podobnego poziomu. Najwięcej było ich w **2025-10 (66)**  a najmniej w **2025-08 (33)**; różnice wynikają głównie z losowego generowania dat.
5. Cena biletu praktycznie nie jest związana z wypełnieniem sali (korelacja **0.009**).
